<a href="https://colab.research.google.com/github/gd-Sahat/ClockBiasPINN/blob/main/tsf_mamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet torch

In [2]:
!apt-get update -qq && apt-get install -qq libomp-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
!pip install mamba-ssm --no-build-isolation

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
DATA_PATH = '/content/drive/MyDrive/CLOCKBIASPINN/clkbias_diff.gz'

In [8]:
df = pd.read_csv(DATA_PATH, compression='gzip')

In [9]:
# If timestamp is not datetime, convert it
if not np.issubdtype(df['timestamp'].dtype, np.datetime64):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
# Create time_ns feature directly from datetime (nanoseconds since epoch)
df['time_ns'] = df['timestamp'].astype(np.int64)
df['hours_since_start'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600 #raw ns cause scaling issue in data

# Encode satellite ID globally using factorize
df['id_enc'] = pd.factorize(df['id'])[0]


In [34]:
# Number of points for your rolling‐slope
N = 32
t = np.arange(N)

# 1) Instantaneous drift (first difference)
df['drift1'] = df['bias_interp_ns'].diff()

# 2) Lagged drift (previous step)
df['drift2'] = df['drift1'].shift(1)


In [35]:
UPDATE_INTERVAL = 7200  # 2 hours in seconds
df['seconds_since_update'] = df['timestamp'].astype(np.int64) // 1e9 % UPDATE_INTERVAL

In [36]:
# Use np.where for element-wise condition checking
df['update_flag'] = np.where(df['seconds_since_update'] % UPDATE_INTERVAL == 0, 1, 0)

In [37]:
df['update_sin'] = np.sin(2 * np.pi * df['seconds_since_update'] / UPDATE_INTERVAL)
df['update_cos'] = np.cos(2 * np.pi * df['seconds_since_update'] / UPDATE_INTERVAL)

In [38]:
# Prepare raw features
raw_features = df[['id_enc','bias_interp_ns','bias_diff_ns','hours_since_start', 'update_flag', 'drift1', 'drift2']].values

# Prepare and standardize targets
targets = df['bias_diff_ns'].values.astype(np.float32).reshape(-1, 1)

In [49]:
df

,id,timestamp,bias_interp_s,bias_s,bias_diff,bias_interp_ns,bias_ns,bias_diff_ns,time_ns,hours_since_start,id_enc,seconds_since_update,update_flag,update_sin,update_cos,drift1,drift2
0,G01,2023-05-01 00:00:00,0.000189,0.000189,-2.555663e-09,188869.424164,188866.868501,-2.555663,1682899200000000000,0.000000,0,0.0,1,0.000000,1.000000,NaN,NaN
1,G01,2023-05-01 00:00:30,0.000189,0.000189,-2.543851e-09,188869.332078,188866.788226,-2.543851,1682899230000000000,0.008333,0,30.0,0,0.026177,0.999657,-0.092086,NaN
2,G01,2023-05-01 00:01:00,0.000189,0.000189,-2.534747e-09,188869.239991,188866.705245,-2.534747,1682899260000000000,0.016667,0,60.0,0,0.052336,0.998630,-0.092086,-0.092086
3,G01,2023-05-01 00:01:30,0.000189,0.000189,-2.547661e-09,188869.147905,188866.600244,-2.547661,1682899290000000000,0.025000,0,90.0,0,0.078459,0.996917,-0.092086,-0.092086
4,G01,2023-05-01 00:02:00,0.000189,0.000189,-2.551347e-09,188869.055819,188866.504471,-2.551347,1682899320000000000,0.033333,0,120.0,0,0.104528,0.994522,-0.092086,-0.092086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66029711,G32,2025-05-01 23:57:30,-0.000463,-0.000463,-1.760195e-09,-463035.542680,-463037.302875,-1.760195,1746143850000000000,17567.958333,31,7050.0,0,-0.130526,0.991445,0.296723,0.296723
66029712,G32,2025-05-01 23:58:00,-0.000463,-0.000463,-1.774284e-09,-463035.245957,-463037.020241,-1.774284,1746143880000000000,17567.966667,31,7080.0,0,-0.104528,0.994522,0.296723,0.296723
66029713,G32,2025-05-01 23:58:30,-0.000463,-0.000463,-1.778575e-09,-463034.949235,-463036.727809,-1.778575,1746143910000000000,17567.975000,31,7110.0,0,-0.078459,0.996917,0.296723,0.296723
66029714,G32,2025-05-01 23:59:00,-0.000463,-0.000463,-1.771546e-09,-463034.652512,-463036.424058,-1.771546,1746143940000000000,17567.983333,31,7140.0,0,-0.052336,0.998630,0.296723,0.296723


In [39]:
'''# --- 3. Scale and Split ---
N = len(raw_features)
train_end = int(N * 0.8)
val_end = int(N * 0.9)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
scaler_X.fit(raw_features[:train_end])
scaler_y.fit(targets[:train_end].reshape(-1, 1))

features_scaled = scaler_X.transform(raw_features)
targets_scaled = scaler_y.transform(targets.reshape(-1, 1)).ravel()

features_train = features_scaled[:train_end]
targets_train  = targets_scaled[:train_end]
features_val   = features_scaled[train_end:val_end]
targets_val    = targets_scaled[train_end:val_end]
features_test  = features_scaled[val_end:]
targets_test   = targets_scaled[val_end:]'''

'# --- 3. Scale and Split ---\nN = len(raw_features)\ntrain_end = int(N * 0.8)\nval_end = int(N * 0.9)\n\nscaler_X = StandardScaler()\nscaler_y = StandardScaler()\nscaler_X.fit(raw_features[:train_end])\nscaler_y.fit(targets[:train_end].reshape(-1, 1))\n\nfeatures_scaled = scaler_X.transform(raw_features)\ntargets_scaled = scaler_y.transform(targets.reshape(-1, 1)).ravel()\n\nfeatures_train = features_scaled[:train_end]\ntargets_train  = targets_scaled[:train_end]\nfeatures_val   = features_scaled[train_end:val_end]\ntargets_val    = targets_scaled[train_end:val_end]\nfeatures_test  = features_scaled[val_end:]\ntargets_test   = targets_scaled[val_end:]'

In [40]:

class LazyWindowDataset(Dataset):
    def __init__(self, features, targets, window_size):
        self.features = features
        self.targets = targets
        self.window_size = window_size

    def __len__(self):
        return len(self.features) - self.window_size

    def __getitem__(self, idx):
        X = self.features[idx:idx+self.window_size]
        y = self.targets[idx+self.window_size]
        return torch.from_numpy(X).float(), torch.tensor(y).float()

In [50]:
import numpy as np
import torch
from torch.utils.data import DataLoader

# --- Parameters ---
WINDOW_SIZE = 32
BATCH_SIZE  = 2048    # you can lower this if you still OOM
DEBUG_MODE  = True
DEBUG_SIZE  = 40000  # for quick tests

# 0) Optionally truncate for debug
if DEBUG_MODE:
    feats = raw_features[:DEBUG_SIZE]
    targs = targets[:DEBUG_SIZE]
else:
    feats = raw_features
    targs = targets

# 1) Sequential split on raw data
N = len(feats)
train_end = int(N * 0.8)
val_end   = int(N * 0.9)

# 2) Fit scalers on train only, transform each split
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler().fit(feats[:train_end])
scaler_y = StandardScaler().fit(targs[:train_end].reshape(-1,1))

feats_train = scaler_X.transform(feats[:train_end])
targs_train = scaler_y.transform(targs[:train_end].reshape(-1,1)).ravel()

feats_val   = scaler_X.transform(feats[train_end:val_end])
targs_val   = scaler_y.transform(targs[train_end:val_end].reshape(-1,1)).ravel()

feats_test  = scaler_X.transform(feats[val_end:])
targs_test  = scaler_y.transform(targs[val_end:].reshape(-1,1)).ravel()

# 3) Use LazyWindowDataset on each split
train_ds = LazyWindowDataset(feats_train, targs_train, WINDOW_SIZE)
val_ds   = LazyWindowDataset(feats_val,   targs_val,   WINDOW_SIZE)
test_ds  = LazyWindowDataset(feats_test,  targs_test,  WINDOW_SIZE)

# 4) Build loaders (reduce batch size if necessary)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


In [51]:
# right after you instantiate val_loader
xb, yb = next(iter(val_loader))
print(yb[:5])   # do these look ∼0 (mean) with std≈1?  Or thousands of ns?


tensor([0.6965, 0.6918, 0.6927, 0.6913, 0.6927])


In [52]:
# grab one batch from each loader
xb_tr, yb_tr = next(iter(train_loader))
xb_val, yb_val = next(iter(val_loader))
xb_te, yb_te = next(iter(test_loader))

# print out a few values + overall mean/std
for name, yb in [("TRAIN", yb_tr), ("VAL", yb_val), ("TEST", yb_te)]:
    print(f"\n{name} y[:5]:", yb[:5].cpu().numpy())
    print(f"{name} mean: {yb.mean().item():.4f}, std: {yb.std().item():.4f}")



TRAIN y[:5]: [ 0.79372877  1.1898429  -1.2718709   1.0721759  -0.5655625 ]
TRAIN mean: 0.0064, std: 1.0027

VAL y[:5]: [0.6965222  0.6918192  0.69271547 0.6912828  0.69265044]
VAL mean: 0.1076, std: 0.3698

TEST y[:5]: [0.01563519 0.01662026 0.01576485 0.01668234 0.01550161]
TEST mean: -0.0336, std: 0.2713


In [53]:
print("Num training windows:", len(train_ds))
print("Batches per train epoch:", len(train_loader))
print("Num val windows:", len(val_ds))
print("Batches per val epoch:", len(val_loader))
print("Num test windows:", len(test_ds))
print("Batches per test epoch:", len(test_loader))

Num training windows: 31965
Batches per train epoch: 16
Num val windows: 3968
Batches per val epoch: 2
Num test windows: 3968
Batches per test epoch: 2


## **With Optimization**

In [54]:
'''import torch.optim as optim
import optuna

class EnhancedMambaModel(nn.Module):
    def __init__(self, input_dim, d_model=64, n_layers=3, dropout=0.05):
        super().__init__()
        self.fc_in = nn.Linear(input_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=d_model, d_state=16)
            for _ in range(n_layers)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.fc_in(x)
        h = self.dropout(h)
        for layer in self.mamba_layers:
            h = layer(h) + h
            h = self.dropout(h)
        h = self.ln(h)
        return self.fc_out(h.mean(dim=1))

# --- Dataset Definition ---
class LazyWindowDataset(torch.utils.data.Dataset):
    def __init__(self, features, targets, window_size):
        self.features = features
        self.targets = targets
        self.window_size = window_size
    def __len__(self):
        return len(self.features) - self.window_size
    def __getitem__(self, idx):
        X = self.features[idx:idx+self.window_size]
        y = self.targets[idx+self.window_size]
        return torch.from_numpy(X).float(), torch.tensor(y).float()

# --- Weight Initialization ---
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)

# --- Optuna Objective ---
def objective(trial):
    # Sample hyperparameters
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    dropout = trial.suggest_uniform('dropout', 0.0, 0.3)
    d_model = trial.suggest_categorical('d_model', [32, 64, 128])
    n_layers = trial.suggest_int('n_layers', 1, 5)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048])

    # Build model
    input_dim = raw_features.shape[1]
    model = EnhancedMambaModel(input_dim=input_dim, d_model=d_model, n_layers=n_layers, dropout=dropout).to(device)
    model.apply(init_weights)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    # Rebuild data loaders with new batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, pin_memory=True)

    best_val_loss = float('inf')
    for epoch in range(1, 10):  # tune for 5 epochs
        # Training
        model.train()
        total_train = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(Xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            total_train += loss.item() * Xb.size(0)
        # Validation
        model.eval()
        total_val = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb).squeeze()
                loss = criterion(preds, yb)
                total_val += loss.item() * Xb.size(0)
        val_loss = total_val / len(val_loader.dataset)

        scheduler.step(val_loss)
        best_val_loss = min(best_val_loss, val_loss)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

# --- Main Execution ---
if __name__ == '__main__':
    # Reproducibility
    SEED = 42
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Prepare raw_features, targets and initial datasets
    # raw_features, targets defined earlier
    # Build train_ds, val_ds once with a baseline split
    train_ds = LazyWindowDataset(feats_train, targs_train, WINDOW_SIZE)
    val_ds   = LazyWindowDataset(feats_val,   targs_val,   WINDOW_SIZE)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print('Best hyperparams:', study.best_trial.params)
    print('Best val MSE:', study.best_trial.value)
'''

"import torch.optim as optim\nimport optuna\n\nclass EnhancedMambaModel(nn.Module):\n    def __init__(self, input_dim, d_model=64, n_layers=3, dropout=0.05):\n        super().__init__()\n        self.fc_in = nn.Linear(input_dim, d_model)\n        self.dropout = nn.Dropout(dropout)\n        self.mamba_layers = nn.ModuleList([\n            Mamba(d_model=d_model, d_state=16)\n            for _ in range(n_layers)\n        ])\n        self.ln = nn.LayerNorm(d_model)\n        self.fc_out = nn.Linear(d_model, 1)\n\n    def forward(self, x):\n        h = self.fc_in(x)\n        h = self.dropout(h)\n        for layer in self.mamba_layers:\n            h = layer(h) + h\n            h = self.dropout(h)\n        h = self.ln(h)\n        return self.fc_out(h.mean(dim=1))\n\n# --- Dataset Definition ---\nclass LazyWindowDataset(torch.utils.data.Dataset):\n    def __init__(self, features, targets, window_size):\n        self.features = features\n        self.targets = targets\n        self.window_size 

In [55]:
'''import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice
)
# 1. How the best value evolved over trials
fig1 = plot_optimization_history(study)
fig1.show()

# 2. Which hyperparameters had the biggest impact
fig2 = plot_param_importances(study)
fig2.show()

# 3. Interactions between two or more parameters
fig3 = plot_parallel_coordinate(study, params=['lr', 'weight_decay', 'dropout'])
fig3.show()

# 4. Slice plots to see performance vs. a single parameter
fig4 = plot_slice(study, params=['lr', 'd_model', 'weight_decay', 'dropout'])
fig4.show()
'''

"import optuna\nfrom optuna.visualization import (\n    plot_optimization_history,\n    plot_param_importances,\n    plot_parallel_coordinate,\n    plot_slice\n)\n# 1. How the best value evolved over trials\nfig1 = plot_optimization_history(study)\nfig1.show()\n\n# 2. Which hyperparameters had the biggest impact\nfig2 = plot_param_importances(study)\nfig2.show()\n\n# 3. Interactions between two or more parameters\nfig3 = plot_parallel_coordinate(study, params=['lr', 'weight_decay', 'dropout'])\nfig3.show()\n\n# 4. Slice plots to see performance vs. a single parameter\nfig4 = plot_slice(study, params=['lr', 'd_model', 'weight_decay', 'dropout'])\nfig4.show()\n"

In [56]:
import time
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

# --- Enhanced Mamba Model with Dropout ---
class EnhancedMambaModel(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=3, dropout=0.1):
        super().__init__()
        self.fc_in = nn.Linear(input_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=d_model, d_state=16)
            for _ in range(n_layers)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.fc_in(x)
        h = self.dropout(h)
        for layer in self.mamba_layers:
            h = layer(h) + h
            h = self.dropout(h)
        h = self.ln(h)
        return self.fc_out(h.mean(dim=1))

# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = feats_train.shape[1]
model = EnhancedMambaModel(input_dim=input_dim, d_model=128, n_layers=3, dropout=0.1).to(device)

# Initialize weights (clean model before training)
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
model.apply(init_weights)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-7)
# LR scheduler to reduce LR on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    verbose=True
)

# Early stopping parameters
epochs = 100
best_val = float("inf")
patience = 50
epochs_no_improve = 0

# --- Training and Validation Loop ---
train_losses = []
val_losses = []
for epoch in range(1, epochs + 1):
    # Training
    model.train()
    total_train = 0
    start = time.time()
    for Xb, yb in tqdm(train_loader, desc=f"Epoch {epoch} [Train]", unit="batch"):
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(Xb).squeeze()
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_train += loss.item() * Xb.size(0)
    train_loss = total_train / len(train_loader.dataset)
    train_losses.append(train_loss)
    print(f"Epoch {epoch}: Train Loss {train_loss:.5f} ({time.time() - start:.2f}s)")

    # Validation
    model.eval()
    total_val = 0
    with torch.no_grad():
        for Xb, yb in tqdm(val_loader, desc=f"Epoch {epoch} [Val]", unit="batch"):
            Xb, yb = Xb.to(device), yb.to(device)
            preds = model(Xb).squeeze()
            loss = criterion(preds, yb)
            total_val += loss.item() * Xb.size(0)
    val_loss = total_val / len(val_loader.dataset)
    val_losses.append(val_loss)
    print(f"Epoch {epoch}: Val   Loss {val_loss:.5f}")

    # Scheduler step
    scheduler.step(val_loss)

    # Early stopping check
    if val_loss < best_val - 1e-6:
        best_val = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"No improvement after {patience} epochs, stopping early.")
            break

# --- Test Evaluation ---
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
test_loss = 0
with torch.no_grad():
    for Xb, yb in tqdm(test_loader, desc="Test Eval", unit="batch"):
        Xb, yb = Xb.to(device), yb.to(device)
        preds = model(Xb).squeeze()
        loss = criterion(preds, yb)
        test_loss += loss.item() * Xb.size(0)
    test_loss /= len(test_loader.dataset)
print(f"Test Loss: {test_loss:.5f}")

# --- Debug: confirm losses stored correctly ---
print(f"Stored {len(train_losses)} train losses and {len(val_losses)} val losses.")
# You can now inspect or plot train_losses and val_losses as needed.


Epoch 1 [Train]: 100%|██████████| 16/16 [00:02<00:00,  5.66batch/s]


Epoch 1: Train Loss nan (2.83s)


Epoch 1 [Val]: 100%|██████████| 2/2 [00:00<00:00, 13.18batch/s]


Epoch 1: Val   Loss nan


Epoch 2 [Train]: 100%|██████████| 16/16 [00:02<00:00,  5.75batch/s]


...

Epoch 5: Train Loss nan (2.80s)


Epoch 5 [Val]: 100%|██████████| 2/2 [00:00<00:00, 13.45batch/s]


Epoch 5: Val   Loss nan


Epoch 6 [Train]:  75%|███████▌  | 12/16 [00:02<00:00,  5.43batch/s]


KeyboardInterrupt: 

In [ ]:
train_losses

In [ ]:
val_losses

In [ ]:
# prompt: plot train_losses and val_losses against epochs

import matplotlib.pyplot as plt

# Plotting the losses
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

# --- Load best model checkpoint ---
model.load_state_dict(torch.load("best_model.pt"))
model.to(device)
model.eval()

# --- Run inference on test_loader ---
preds_scaled = []
trues_scaled = []

with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        # forward
        out = model(Xb).squeeze().cpu().numpy()
        preds_scaled.append(out)
        trues_scaled.append(yb.numpy())

# concatenate batches
preds_scaled = np.concatenate(preds_scaled)   # shape (num_test_windows,)
trues_scaled = np.concatenate(trues_scaled)

# --- Inverse‐transform back to nanoseconds ---
# scaler_y is the StandardScaler you fit on y_train
preds_raw = scaler_y.inverse_transform(preds_scaled.reshape(-1,1)).ravel()
trues_raw = scaler_y.inverse_transform(trues_scaled.reshape(-1,1)).ravel()

# --- Compute and print RMSE ---
rmse = np.sqrt(np.mean((preds_raw - trues_raw)**2))
print(f"Test RMSE: {rmse:.2f} ns")

# Optionally, inspect the first few
for i in range(200):
    print(f"True: {trues_raw[i]:.2f} ns, Pred: {preds_raw[i]:.2f} ns")


In [ ]:
# prompt: plot true vs predicted

# --- Plot True vs Predicted ---
# Using a subset for clarity if the test set is very large
num_samples_to_plot = min(1000, len(trues_raw)) # Plot up to 1000 samples

plt.figure(figsize=(12, 7))
plt.plot(trues_raw[:num_samples_to_plot], label='True Bias Difference (ns)', alpha=0.7)
plt.plot(preds_raw[:num_samples_to_plot], label='Predicted Bias Difference (ns)', alpha=0.7)
plt.title(f'True vs. Predicted Bias Difference (First {num_samples_to_plot} Test Samples)')
plt.xlabel('Sample Index (within test set)')
plt.ylabel('Bias Difference (ns)')
plt.legend()
plt.grid(True)
plt.show()

# Optionally, a scatter plot to show correlation
plt.figure(figsize=(8, 8))
plt.scatter(trues_raw, preds_raw, alpha=0.1, s=10) # Use smaller markers and transparency for large datasets
plt.title('True vs. Predicted Bias Difference (Test Set)')
plt.xlabel('True Bias Difference (ns)')
plt.ylabel('Predicted Bias Difference (ns)')
plt.grid(True)
# Add a diagonal line representing perfect prediction
lims = [
    np.min([plt.xlim(), plt.ylim()]),  # min of both axes
    np.max([plt.xlim(), plt.ylim()]),  # max of both axes
]
plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
plt.xlim(lims)
plt.ylim(lims)
plt.gca().set_aspect('equal', adjustable='box') # Ensure equal scaling
plt.show()
